# Clean movie metadata

Turn the joined Netflix–TMDB table into a raw movie table for later EDA. Keep every matched movie, parse list fields, and store log copies beside the raw numbers. This notebook does not drop rows and does not build one-hot encodings.


## 1. Load metadata

Read the joined movie table written by notebook 01.


In [12]:
import ast
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path("..")
PROCESSED = ROOT / "data" / "processed"
NETFLIX_END_YEAR = 2005


In [13]:
movies = pd.read_csv(PROCESSED / "movies_metadata.csv")
print(movies.shape)
movies.head(2)


(1806, 25)


,netflix_movie_id,tmdb_id,netflix_title,tmdb_title,year,budget,genres,homepage,keywords,original_language,...,release_date,revenue,runtime,spoken_languages,status,tagline,vote_average,vote_count,cast,crew
0,3,17139,Character,Character,1997.0,4500000,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 10769, ""n...",NaN,"[{""id"": 3683, ""name"": ""law and ethics""}]",nl,...,1997-04-17,0,122.0,"[{""iso_639_1"": ""nl"", ""name"": ""Nederlands""}, {""...",Released,NaN,7.7,25,"[{""cast_id"": 4, ""character"": ""Dreverhaven"", ""c...","[{""credit_id"": ""52fe47099251416c7508bca3"", ""de..."
1,30,6964,Something's Gotta Give,Something's Gotta Give,2003.0,80000000,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 35, ""name...",NaN,"[{""id"": 965, ""name"": ""age difference""}, {""id"":...",en,...,2003-12-12,266728738,128.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,Schmucks are people too.,6.3,410,"[{""cast_id"": 1, ""character"": ""Harry Sanborn"", ...","[{""credit_id"": ""548200699251416e7800666d"", ""de..."


## 2. Drop unused columns and unify the title

Drop sparse or unused text fields. Keep both ids and a single title.


In [14]:
DROP_COLS = [
    "homepage",
    "tagline",
    "keywords",
    "status",
    "popularity",
    "vote_average",
    "vote_count",
    "tmdb_title",
    "original_title",
    "overview",
    "release_date",
]
df = movies.drop(columns=DROP_COLS).rename(columns={"netflix_title": "title"})
print(df.shape)
print(list(df.columns))


(1806, 14)
['netflix_movie_id', 'tmdb_id', 'title', 'year', 'budget', 'genres', 'original_language', 'production_companies', 'production_countries', 'revenue', 'runtime', 'spoken_languages', 'cast', 'crew']


## 3. Parse JSON fields

Turn genre, company, country, language, and cast strings into name lists. Cast names stay in billing order. Extract the director from the crew list, then drop the raw JSON columns.


In [15]:
def parse_object(val):
    if pd.isna(val) or val == "":
        return []
    try:
        obj = ast.literal_eval(val) if isinstance(val, str) else val
    except (ValueError, SyntaxError):
        return []
    return obj if isinstance(obj, list) else []


def parse_names(val):
    return [x.get("name") for x in parse_object(val) if isinstance(x, dict) and x.get("name")]


def parse_cast(val):
    people = [x for x in parse_object(val) if isinstance(x, dict) and x.get("name")]
    people.sort(key=lambda person: person.get("order", 999))
    return [person["name"] for person in people]


def extract_director(crew_str):
    for person in parse_object(crew_str):
        if isinstance(person, dict) and person.get("job") == "Director":
            return person.get("name")
    return None


In [16]:
df["genre_names"] = df["genres"].map(parse_names)
df["company_names"] = df["production_companies"].map(parse_names)
df["country_names"] = df["production_countries"].map(parse_names)
df["language_names"] = df["spoken_languages"].map(parse_names)
df["cast_names"] = df["cast"].map(parse_cast)
df["director"] = df["crew"].map(extract_director)
df = df.drop(columns=["genres", "production_companies", "production_countries", "spoken_languages", "cast", "crew"])

print("directors missing:", int(df["director"].isna().sum()))
print(df["cast_names"].map(len).describe()[["mean", "50%", "max"]].to_dict())
df.head(3)


directors missing: 2
{'mean': 21.696013289036546, '50%': 16.0, 'max': 213.0}


,netflix_movie_id,tmdb_id,title,year,budget,original_language,revenue,runtime,genre_names,company_names,country_names,language_names,cast_names,director
0,3,17139,Character,1997.0,4500000,nl,0,122.0,"[Drama, Foreign, History]","[First Floor Features, Almerica Films]","[Netherlands, Belgium]","[Nederlands, English, Français, Deutsch]","[Jan Decleir, Fedja van Huêt, Betty Schuurman,...",Mike van Diem
1,30,6964,Something's Gotta Give,2003.0,80000000,en,266728738,128.0,"[Drama, Comedy, Romance]","[Columbia Pictures Corporation, Waverly Films,...",[United States of America],"[Français, English]","[Jack Nicholson, Diane Keaton, Keanu Reeves, F...",Nancy Meyers
2,55,11863,Jade,1995.0,50000000,en,9851610,95.0,"[Action, Thriller, Mystery, Romance]",[Paramount Pictures],[United States of America],[English],"[David Caruso, Linda Fiorentino, Chazz Palmint...",William Friedkin


## 4. Add age, logs, and ROI

`movie_age` is years before the end of the Netflix ratings (2005). Log copies use `log1p` so zeros stay finite. ROI is `revenue / budget`, and `log_roi` is `log(revenue) - log(budget)`, only where both amounts are positive. Zeros stay in the table.


In [17]:
df["movie_age"] = NETFLIX_END_YEAR - df["year"]
for raw_col, log_col in [
    ("budget", "budget_log"),
    ("revenue", "revenue_log"),
    ("runtime", "runtime_log"),
]:
    df[log_col] = np.log1p(df[raw_col])

# log1p is undefined for a negative age (a film released after 2005)
df["movie_age_log"] = np.where(df["movie_age"] >= 0, np.log1p(df["movie_age"]), np.nan)

positive_money = (df["budget"] > 0) & (df["revenue"] > 0)
df["roi"] = np.where(positive_money, df["revenue"] / df["budget"], np.nan)
df["log_roi"] = np.where(
    positive_money, np.log(df["revenue"]) - np.log(df["budget"]), np.nan
)

print("movies with year after 2005:", int((df["movie_age"] < 0).sum()))
df[["budget", "budget_log", "revenue", "revenue_log", "roi", "log_roi", "movie_age", "movie_age_log"]].describe()


movies with year after 2005: 0


c:\Users\Revital\Documents\לימודים\Project\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,budget,budget_log,revenue,revenue_log,roi,log_roi,movie_age,movie_age_log
count,1.806000e+03,1806.000000,1.806000e+03,1806.000000,1.322000e+03,1322.000000,1806.000000,1806.000000
mean,2.794110e+07,13.908393,7.320716e+07,13.136488,6.439837e+03,0.803666,10.320044,2.013569
std,3.169984e+07,6.369681,1.220280e+08,7.721064,2.337775e+05,1.588656,11.869302,0.902307
min,0.000000e+00,0.000000,0.000000e+00,0.000000,5.217391e-07,-14.466098,0.000000,0.000000
25%,3.000000e+06,14.914123,1.775000e+01,2.919423,1.072081e+00,0.069602,3.000000,1.386294
50%,1.800000e+07,16.705882,2.672748e+07,17.101201,2.336906e+00,0.848827,6.000000,1.945910
75%,4.275000e+07,17.570828,9.488396e+07,18.368165,4.769094e+00,1.562155,12.000000,2.564949
max,2.000000e+08,19.113828,1.845034e+09,21.335764,8.500000e+06,15.955577,89.000000,4.499810


## 5. Save the raw table

List columns are stored as JSON text. The file keeps every joined movie and both the raw numbers and their log copies.


In [18]:
LIST_COLS = [
    "genre_names",
    "company_names",
    "country_names",
    "language_names",
    "cast_names",
]
for col in LIST_COLS:
    df[col] = df[col].map(json.dumps)

KEEP = [
    "netflix_movie_id",
    "tmdb_id",
    "title",
    "year",
    "original_language",
    "budget",
    "revenue",
    "runtime",
    "movie_age",
    "budget_log",
    "revenue_log",
    "runtime_log",
    "movie_age_log",
    "roi",
    "log_roi",
    "director",
    *LIST_COLS,
]
cleaned = df[KEEP].copy()
print("rows:", len(cleaned), "cols:", cleaned.shape[1])
print("nulls:", int(cleaned.drop(columns=["director", "roi", "log_roi", "movie_age_log"]).isna().sum().sum()))
cleaned.head(2)


rows: 1806 cols: 21
nulls: 0


,netflix_movie_id,tmdb_id,title,year,original_language,budget,revenue,runtime,movie_age,budget_log,...,runtime_log,movie_age_log,roi,log_roi,director,genre_names,company_names,country_names,language_names,cast_names
0,3,17139,Character,1997.0,nl,4500000,0,122.0,8.0,15.319588,...,4.812184,2.197225,NaN,NaN,Mike van Diem,"[""Drama"", ""Foreign"", ""History""]","[""First Floor Features"", ""Almerica Films""]","[""Netherlands"", ""Belgium""]","[""Nederlands"", ""English"", ""Fran\u00e7ais"", ""De...","[""Jan Decleir"", ""Fedja van Hu\u00eat"", ""Betty ..."
1,30,6964,Something's Gotta Give,2003.0,en,80000000,266728738,128.0,2.0,18.197537,...,4.859812,1.098612,3.334109,1.204206,Nancy Meyers,"[""Drama"", ""Comedy"", ""Romance""]","[""Columbia Pictures Corporation"", ""Waverly Fil...","[""United States of America""]","[""Fran\u00e7ais"", ""English""]","[""Jack Nicholson"", ""Diane Keaton"", ""Keanu Reev..."


In [19]:
out_path = PROCESSED / "movies_cleaned.csv"
cleaned.to_csv(out_path, index=False)
print(f"saved {out_path} ({cleaned.shape[0]} rows x {cleaned.shape[1]} cols)")


saved ..\data\processed\movies_cleaned.csv (1806 rows x 21 cols)
